<a href="https://colab.research.google.com/github/nermal1/Stock-Market-Prediction-437/blob/main/SVMRealPred.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Import Libraries

In [1]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from itertools import combinations

from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score

## Setup

In [2]:
tickers = ['AAPL', 'MSFT', '^GSPC', '^DJI']
results = {}

## Indicator Functions

In [3]:
def weighted_moving_average(data, period):
  weights = np.arange(1, period + 1)
  return data.rolling(period).apply(lambda x: np.dot(x, weights) / weights.sum(), raw=True)

In [4]:
def featureSelection(df):
  df = df.copy()

  # Basic Returns
  df['Return'] = df['Close'].pct_change()

  # Technical Indicators
  df['SMA_14'] = df['Close'].rolling(window=14).mean()
  df['SMA_50'] = df['Close'].rolling(window=50).mean()
  df['WMA_14'] = weighted_moving_average(df['Close'], 14)
  df['Momentum_10'] = df['Close'] / df['Close'].shift(10) - 1
  df['Volatility_14'] = df['Return'].rolling(window=14).std()

  # RSI Calculation
  delta = df['Close'].diff()
  gain = (delta.where(delta > 0, 0))
  loss = (-delta.where(delta < 0, 0))
  avg_gain = gain.rolling(window=14).mean()
  avg_loss = loss.rolling(window=14).mean()
  rs = avg_gain / avg_loss
  df['RSI_14'] = 100 - (100 / (1 + rs))

  # Lags (Previous days' returns)
  lags = [1, 2, 3, 5]
  for lag in lags:
      df[f'Lag_{lag}'] = df['Return'].shift(lag)

  # Target: 1 if Up, 0 if Down
  df['Target'] = np.where(df['Return'] > 0, 1, 0)

  return df.dropna()


## Data pipeline

In [5]:
for ticker in tickers:
  print(f"Fetching data for {ticker}...")
  raw_df = yf.download(ticker, start="2010-01-01", end="2019-12-31", progress=False, auto_adjust=True)

  if isinstance(raw_df.columns, pd.MultiIndex):
      raw_df = raw_df.xs(ticker, axis=1, level=1)

  results[ticker] = featureSelection(raw_df)

Fetching data for AAPL...
Fetching data for MSFT...
Fetching data for ^GSPC...
Fetching data for ^DJI...


## SVM optimization loop

In [9]:
print("SVM Optimization of C and Features")

feature_pool = ['RSI_14', 'SMA_14', 'SMA_50', 'WMA_14', 'Momentum_10', 'Volatility_14', 'Lag_1', 'Lag_2', 'Lag_5']
C_values = [0.5, 1, 5, 10]

for ticker, df in results.items():
  y = df['Target']

  best_accuracy = 0
  best_combo = []
  best_C = 1.0
  best_metrics = {}

  for r in range(1, 6):
    for combo in combinations(feature_pool, r):
      combo_list = list(combo)
      X = df[combo_list]

        # Split Data (Time Series Split)
      X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

        # Scale Data
      scaler = StandardScaler()
      X_train_scaled = scaler.fit_transform(X_train)
      X_test_scaled = scaler.transform(X_test)

        # Loop through C values
      for c_val in C_values:
        model = SVC(kernel='rbf', C=c_val)
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)

        acc = accuracy_score(y_test, y_pred)

        if acc > best_accuracy:
          best_accuracy = acc
          best_combo = combo_list
          best_C = c_val
          best_metrics = {
            'Precision': precision_score(y_test, y_pred, zero_division=0),
            'Recall': recall_score(y_test, y_pred, zero_division=0),
            'F1': f1_score(y_test, y_pred, zero_division=0)
        }

  print(f"\nResults for {ticker}:")
  print(f"  Best Accuracy: {best_accuracy:.2%}")
  print(f"  Best C Value:  {best_C}")
  print(f"  Best Features: {best_combo}")
  print(f"  Precision:     {best_metrics['Precision']:.4f}")
  print(f"  Recall:        {best_metrics['Recall']:.4f}")
  print(f"  F1 Score:      {best_metrics['F1']:.4f}")

SVM Optimization of C and Features

Results for AAPL:
  Best Accuracy: 64.78%
  Best C Value:  0.5
  Best Features: ['Momentum_10', 'Lag_1', 'Lag_2']
  Precision:     0.6448
  Recall:        0.7970
  F1 Score:      0.7129

Results for MSFT:
  Best Accuracy: 63.77%
  Best C Value:  0.5
  Best Features: ['RSI_14', 'Momentum_10', 'Lag_1', 'Lag_2', 'Lag_5']
  Precision:     0.6431
  Recall:        0.8107
  F1 Score:      0.7172

Results for ^GSPC:
  Best Accuracy: 65.18%
  Best C Value:  5
  Best Features: ['Momentum_10', 'Lag_1', 'Lag_2', 'Lag_5']
  Precision:     0.6441
  Recall:        0.8321
  F1 Score:      0.7261

Results for ^DJI:
  Best Accuracy: 64.57%
  Best C Value:  1
  Best Features: ['Momentum_10', 'Volatility_14', 'Lag_1', 'Lag_2', 'Lag_5']
  Precision:     0.6502
  Recall:        0.7721
  F1 Score:      0.7059
